In [1]:
import torch

In [2]:
def ppo_clip_loss(old_log_probs,new_log_probs,advantages,clip_eps=0.2):
  old_log_probs = old_log_probs.detach()
  advantages = advantages.detach()
  ratio = torch.exp(new_log_probs - old_log_probs)
  surr1 = ratio * advantages
  clipped_ratio = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
  surr2 = clipped_ratio * advantages
  surr_min = torch.min(surr1, surr2)
  loss = -surr_min.mean()
  return loss

In [3]:
def build_inputs(ratio,adv,requires_grad=True):
  old_lp = torch.tensor([-2.0])
  new_lp = old_lp + torch.log(torch.tensor([ratio]))

  if requires_grad:
      new_lp.requires_grad_(True)
  adv_tensor = torch.tensor([adv])
  return old_lp,new_lp,adv_tensor

In [6]:
def test_ratio_equals_one():
  ratio=1.0
  adv=1.0
  loss = ppo_clip_loss()
  assert torch.isclose(loss,torch.tensor(-1.0))
  print("✅ test_ratio_equals_one 通过")

In [7]:
import torch

# ================= 1. 核心 Loss 函数 =================
def ppo_clip_loss(old_log_probs, new_log_probs, advantages, clip_eps=0.2):
    old_log_probs = old_log_probs.detach()
    advantages = advantages.detach()
    ratio = torch.exp(new_log_probs - old_log_probs)
    surr1 = ratio * advantages
    clipped_ratio = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
    surr2 = clipped_ratio * advantages
    surr_min = torch.min(surr1, surr2)
    loss = -surr_min.mean()
    return loss

# ================= 2. 构造输入的工具函数 =================
def build_inputs(ratio, adv, requires_grad=True):
    old_lp = torch.tensor([-2.0])
    # 按照题目要求构造 new_lp
    new_lp = old_lp + torch.log(torch.tensor([ratio]))

    # 为了让梯度能往回传，我们需要确保 new_lp 参与计算图
    # 由于 new_lp 是计算出来的（非叶子节点），为了测试，我们直接将其设置需要梯度
    if requires_grad:
        new_lp.requires_grad_(True)

    adv_tensor = torch.tensor([adv])
    return old_lp, new_lp, adv_tensor

# ================= 3. 四个测试函数 =================

def test_ratio_equals_one():
    """测试 Ratio=1.0, Adv=1.0 -> 预期目标 1.0 (Loss -1.0)"""
    old_lp, new_lp, adv_t = build_inputs(ratio=1.0, adv=1.0)
    loss = ppo_clip_loss(old_lp, new_lp, adv_t)
    # 注意：Loss 是带负号的，所以预期是 -1.0
    assert torch.isclose(loss, torch.tensor(-1.0)), f"预期 -1.0, 实际 {loss.item()}"
    print("✅ test_ratio_equals_one 通过")

def test_positive_advantage_clip():
    """测试正向优势，验证截断生效"""
    # 场景1：Ratio=0.5 (未触发截断，0.5 < 1.2) -> 预期目标 0.5 (Loss -0.5)
    old_lp, new_lp, adv_t = build_inputs(ratio=0.5, adv=1.0)
    loss1 = ppo_clip_loss(old_lp, new_lp, adv_t)
    assert torch.isclose(loss1, torch.tensor(-0.5)), f"预期 -0.5, 实际 {loss1.item()}"

    # 场景2：Ratio=1.5 (触发截断，被限制在 1.2) -> 预期目标 1.2 (Loss -1.2)
    old_lp, new_lp, adv_t = build_inputs(ratio=1.5, adv=1.0)
    loss2 = ppo_clip_loss(old_lp, new_lp, adv_t)
    assert torch.isclose(loss2, torch.tensor(-1.2)), f"预期 -1.2, 实际 {loss2.item()}"
    print("✅ test_positive_advantage_clip 通过")

def test_negative_advantage_clip():
    """测试负向优势，验证截断生效"""
    # 场景1：Ratio=1.5 (未触发截断，1.5 * -1.0 = -1.5) -> 预期目标 -1.5 (Loss 1.5)
    old_lp, new_lp, adv_t = build_inputs(ratio=1.5, adv=-1.0)
    loss1 = ppo_clip_loss(old_lp, new_lp, adv_t)
    assert torch.isclose(loss1, torch.tensor(1.5)), f"预期 1.5, 实际 {loss1.item()}"

    # 场景2：Ratio=0.5 (触发截断，被限制在 0.8) -> 预期目标 -0.8 (Loss 0.8)
    old_lp, new_lp, adv_t = build_inputs(ratio=0.5, adv=-1.0)
    loss2 = ppo_clip_loss(old_lp, new_lp, adv_t)
    assert torch.isclose(loss2, torch.tensor(0.8)), f"预期 0.8, 实际 {loss2.item()}"
    print("✅ test_negative_advantage_clip 通过")

def test_gradient_flow():
    """测试梯度流，确保只有新策略有梯度，旧策略和优势没有"""
    old_lp, new_lp, adv_t = build_inputs(ratio=1.2, adv=1.0)

    loss = ppo_clip_loss(old_lp, new_lp, adv_t)
    loss.backward()

    # 检查梯度
    assert new_lp.grad is not None, "new_lp 没有梯度！"
    assert old_lp.grad is None, "old_lp 不应该有梯度！"
    assert adv_t.grad is None, "adv_t 不应该有梯度！"
    print("✅ test_gradient_flow 通过")

# ================= 4. 运行所有测试 =================
if __name__ == "__main__":
    print("开始单元测试...")
    test_ratio_equals_one()
    test_positive_advantage_clip()
    test_negative_advantage_clip()
    test_gradient_flow()
    print("🎉 所有测试全部通过！你的 PPO Loss 实现非常完美！")

开始单元测试...
✅ test_ratio_equals_one 通过
✅ test_positive_advantage_clip 通过
✅ test_negative_advantage_clip 通过
✅ test_gradient_flow 通过
🎉 所有测试全部通过！你的 PPO Loss 实现非常完美！
